#**Road Segmentation from Satellite Images**

In [ ]:
# Cloning the repository in the notebook
import os

REPO_OWNER = "Epoch-2026-2027"
REPO_NAME = "Learning_Phase"
BRANCH_NAME = "Utkarsh"

REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

if not os.path.exists(REPO_NAME):
    !git clone -b {BRANCH_NAME} {REPO_URL}
else:
    print(f"Directory '{REPO_NAME}' already exists.")

os.chdir(REPO_NAME)
print(f"Current working directory changed to: {os.getcwd()}")

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Importing the necessary libraries

import numpy as np
import math
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import os
import shutil
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

##**Data Pre-Processing:**
The pre-processing code has been commented out since it only had to be run once. The processed images were saved and are then loaded into the notebook. In the interest of memory limitations, the original 1500 x 1500 resolution images in the dataset were downscaled to 256 x 256. The models were then built to handle this 256 x 256 resolution.

In [ ]:
"""
# Copying the dataset to local storage
# LLM used here for the copying code

image_dir = '/content/drive/MyDrive/Datasets/Epoch/Massachusetts_Roads_Dataset/'
local_dir = '/content/local_dataset/'

# Gather all files to copy recursively
files_to_copy = []
for root, dirs, files in os.walk(image_dir):
    for file in files:
        files_to_copy.append(os.path.join(root, file))

# Copy files while tracking progress
for src_file in tqdm(files_to_copy, desc="Copying Dataset to Local Storage"):
    # Recreate the directory substructure locally
    rel_path = os.path.relpath(src_file, image_dir)
    dst_file = os.path.join(local_dir, rel_path)
    os.makedirs(os.path.dirname(dst_file), exist_ok=True)

    shutil.copy2(src_file, dst_file)
"""

In [ ]:
"""
# Setup paths
image_folder_path = '/content/local_dataset/'
train_input_folder_path = 'train/'
train_labels_folder_path = 'train_labels/'
output_drive_dir = '/content/drive/MyDrive/Datasets/Epoch/Massachusetts_Roads_Processed/train_batches/'

os.makedirs(output_drive_dir, exist_ok=True)

# Gather and sort file list
train_labels_folder_dir = sorted(os.listdir(os.path.join(image_folder_path, train_labels_folder_path)))

# Configuration parameters
BATCH_SIZE = 100
total_images = len(train_labels_folder_dir)

# LLM used here to set up the processing loops
# for the training, validation and test images
# Outer loop iterating through chunks of files
for i in range(0, total_images, BATCH_SIZE):
    batch_files = train_labels_folder_dir[i:i + BATCH_SIZE]
    batch_num = (i // BATCH_SIZE) + 1

    batch_images = []
    batch_labels = []

    # Inner loop with visual bar for tracking the current segment
    for image_name in tqdm(batch_files, desc=f"Processing Train Batch {batch_num}"):
        image_path = os.path.join(image_folder_path, train_input_folder_path, f'{image_name}f')
        label_path = os.path.join(image_folder_path, train_labels_folder_path, image_name)

        with Image.open(image_path).convert('RGB') as img:
            img_resized = img.resize((256, 256), Image.Resampling.BILINEAR)
            batch_images.append(np.array(img_resized, dtype=np.uint8))

        with Image.open(label_path).convert('L') as lbl:
            lbl_resized = lbl.resize((256, 256), Image.Resampling.NEAREST)
            batch_labels.append(np.array(lbl_resized, dtype=np.uint8))

    # Package individual segment to NumPy master arrays
    X_batch = np.array(batch_images)
    y_batch = np.array(batch_labels)

    # Save the compressed chunk instantly to persistent storage
    batch_save_path = os.path.join(output_drive_dir, f'train_batch_{batch_num}.npz')
    np.savez_compressed(batch_save_path, input=X_batch, labels=y_batch)

    print(f"Saved batch {batch_num} to Drive: {X_batch.shape}")

    # Clean memory explicitly
    del batch_images, batch_labels, X_batch, y_batch

# Setup paths
val_input_folder_path = 'val/'
val_labels_folder_path = 'val_labels/'
output_drive_dir = '/content/drive/MyDrive/Datasets/Epoch/Massachusetts_Roads_Processed/val_batches/'

os.makedirs(output_drive_dir, exist_ok=True)

# Gather and sort file list
val_labels_folder_dir = sorted(os.listdir(os.path.join(image_folder_path, val_labels_folder_path)))

# Configuration parameters
BATCH_SIZE = 100
total_images = len(val_labels_folder_dir)

# Outer loop iterating through chunks of files
for i in range(0, total_images, BATCH_SIZE):
    batch_files = val_labels_folder_dir[i:i + BATCH_SIZE]
    batch_num = (i // BATCH_SIZE) + 1

    batch_images = []
    batch_labels = []

    # Inner loop with visual bar for tracking the current segment
    for image_name in tqdm(batch_files, desc=f"Processing Val Batch {batch_num}"):
        image_path = os.path.join(image_folder_path, val_input_folder_path, f'{image_name}f')
        label_path = os.path.join(image_folder_path, val_labels_folder_path, image_name)

        with Image.open(image_path).convert('RGB') as img:
            img_resized = img.resize((256, 256), Image.Resampling.BILINEAR)
            batch_images.append(np.array(img_resized, dtype=np.uint8))

        with Image.open(label_path).convert('L') as lbl:
            lbl_resized = lbl.resize((256, 256), Image.Resampling.NEAREST)
            batch_labels.append(np.array(lbl_resized, dtype=np.uint8))

    # Package individual segment to NumPy master arrays
    X_batch = np.array(batch_images)
    y_batch = np.array(batch_labels)

    # Save the compressed chunk instantly to persistent storage
    batch_save_path = os.path.join(output_drive_dir, f'val_batch_{batch_num}.npz')
    np.savez_compressed(batch_save_path, input=X_batch, labels=y_batch)

    print(f"Saved batch {batch_num} to Drive: {X_batch.shape}")

    # Clean memory explicitly
    del batch_images, batch_labels, X_batch, y_batch

# Setup paths
test_input_folder_path = 'test/'
test_labels_folder_path = 'test_labels/'
output_drive_dir = '/content/drive/MyDrive/Datasets/Epoch/Massachusetts_Roads_Processed/test_batches/'

os.makedirs(output_drive_dir, exist_ok=True)

# Gather and sort file list
test_labels_folder_dir = sorted(os.listdir(os.path.join(image_folder_path, test_labels_folder_path)))

# Configuration parameters
BATCH_SIZE = 100
total_images = len(test_labels_folder_dir)

# Outer loop iterating through chunks of files
for i in range(0, total_images, BATCH_SIZE):
    batch_files = test_labels_folder_dir[i:i + BATCH_SIZE]
    batch_num = (i // BATCH_SIZE) + 1

    batch_images = []
    batch_labels = []

    # Inner loop with visual bar for tracking the current segment
    for image_name in tqdm(batch_files, desc=f"Processing Test Batch {batch_num}"):
        image_path = os.path.join(image_folder_path, test_input_folder_path, f'{image_name}f')
        label_path = os.path.join(image_folder_path, test_labels_folder_path, image_name)

        with Image.open(image_path).convert('RGB') as img:
            img_resized = img.resize((256, 256), Image.Resampling.BILINEAR)
            batch_images.append(np.array(img_resized, dtype=np.uint8))

        with Image.open(label_path).convert('L') as lbl:
            lbl_resized = lbl.resize((256, 256), Image.Resampling.NEAREST)
            batch_labels.append(np.array(lbl_resized, dtype=np.uint8))

    # Package individual segment to NumPy master arrays
    X_batch = np.array(batch_images)
    y_batch = np.array(batch_labels)

    # Save the compressed chunk instantly to persistent storage
    batch_save_path = os.path.join(output_drive_dir, f'test_batch_{batch_num}.npz')
    np.savez_compressed(batch_save_path, input=X_batch, labels=y_batch)

    print(f"Saved batch {batch_num} to Drive: {X_batch.shape}")

    # Clean memory explicitly
    del batch_images, batch_labels, X_batch, y_batch
"""

##**Data Exploration:**

In [ ]:
# Loading a batch of training images

X_check = np.load('/content/Learning_Phase/Task-2/Subtask-1/Massachusetts_Roads_Processed/train_batches/train_batch_1.npz', allow_pickle=True)['input']
y_check = np.load('/content/Learning_Phase/Task-2/Subtask-1/Massachusetts_Roads_Processed/train_batches/train_batch_1.npz', allow_pickle=True)['labels']

In [ ]:
# Checking the image resolution
print(X_check[0].shape)

In [ ]:
# Checking the average class imbalance (non-road vs road) in the grayscale
# label images across the batch

# Calculating size parameters
pixels_per_image = y_check.shape[1]*y_check.shape[2]
batch_size = y_check.shape[0]

net_road_class_pct = 0
net_non_road_class_pct = 0
for label in y_check:
  road_pct = 0
  non_road_pct = 0
  for row in label:
    for pixel in row:
      # Updating the road vs non-road pixel percentages
      # for a single image
      if int(pixel) == 0:
        non_road_pct += ((1/pixels_per_image)*100)
      elif int(pixel) == 255:
        road_pct += ((1/pixels_per_image)*100)
  # Updating the road vs non-road pixel percentages
  # for the entire batch
  net_road_class_pct += (road_pct/batch_size)
  net_non_road_class_pct += (non_road_pct/batch_size)

print(f"Net Road Class Percentage: {net_road_class_pct:.2f} %")
print(f"Net Non-Road Class Percentage: {net_non_road_class_pct:.2f} %")

## **Baseline UNet Architecture:**

In [ ]:
# Wrapping the image data into datasets
import os

# Setup paths
image_folder_path = '/content/Learning_Phase/Task-2/Subtask-1/Massachusetts_Roads_Processed/'
train_files = [os.path.join(image_folder_path, 'train_batches', f) for f in sorted(os.listdir(os.path.join(image_folder_path, 'train_batches')))]
val_files   = [os.path.join(image_folder_path, 'val_batches', f)   for f in sorted(os.listdir(os.path.join(image_folder_path, 'val_batches')))]
test_files  = [os.path.join(image_folder_path, 'test_batches', f)  for f in sorted(os.listdir(os.path.join(image_folder_path, 'test_batches')))]

# LLM used here to help set up the datasets
class ImageDataset(Dataset):
  def __init__(self, file_paths):
    self.file_paths = file_paths
    # Handling file locations for the
    # getitem method using an internal index map
    self.index_map = []
    # Caching individual batches for efficient GPU processing
    self.cached_data = {}

    # Pre-load/cache files to avoid repetitive disk I/O
    for f_path in self.file_paths:
      with np.load(f_path, allow_pickle=True) as data:
          # Cache arrays in memory to keep Google Drive happy
          self.cached_data[f_path] = {
              'input': data['input'],
              'labels': data['labels']
          }
          num_samples = len(data['input'])

      for i in range(num_samples):
          self.index_map.append((f_path, i))

  def __len__(self):
    return len(self.index_map)

  def __getitem__(self, index):
    file_path, inner_idx = self.index_map[index]

    # Read from memory cache instead of opening disk file
    X_sample = self.cached_data[file_path]['input'][inner_idx]
    y_sample = self.cached_data[file_path]['labels'][inner_idx]

    # Convert and fix dimensions here (H, W, C) -> (C, H, W)
    X_tensor = torch.from_numpy(X_sample).float().permute(2, 0, 1) / 255.0
    # Convert mask values from [0, 255] to [0, 1]
    y_tensor = (torch.from_numpy(y_sample) > 127).long()

    return X_tensor, y_tensor

# Defining the batch size
BATCH_SIZE = 8

# Creating the Datasets and DataLoaders
train_dataset = ImageDataset(train_files)
val_dataset   = ImageDataset(val_files)
test_dataset  = ImageDataset(test_files)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# Baseline UNet Architecture

class UNet(nn.Module):
  def __init__(self, channels, height, width):
    super().__init__()

    # Initializing the UNet parameters
    self.channels = channels
    self.height = height
    self.width = width

    # Defining the blocks of the Contracting Path:
    self.contract = nn.ModuleList([
        # Block 1
        nn.Sequential(
            nn.Conv2d(self.channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        ),
        # Block 2
        nn.Sequential(
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        ),
        # Block 3
        nn.Sequential(
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        ),
        # Block 4
        nn.Sequential(
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU()
        )
    ])

    # Defining the blocks of the Expanding Path:
    self.expand = nn.ModuleList([
        # Block 4
        nn.Sequential(
            nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2),
            nn.BatchNorm2d(256),
            nn.ReLU()
        ),
        # Block 3
        nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2),
            nn.BatchNorm2d(128),
            nn.ReLU()
        ),
        # Block 2
        nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),
            nn.BatchNorm2d(64),
            nn.ReLU()
        ),
        # Block 1
        nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 1, kernel_size=1)
        )
    ])

  def forward(self, x):
    skip_connections = []

    # Passing the input through the contracting path
    # and storing the skip connection information
    for block in self.contract:
      x = block(x)
      skip_connections.append(x)

    # Reversing the skip connections and dropping the last tensor
    # in order to properly align with the expanding path tensors
    skip_connections = list(reversed(skip_connections[:-1]))

    # Passing the encoded input through the expanding path
    # and concatenating with the skip connection tensors
    for idx, block in enumerate(self.expand):
      if idx == 0:
        x = block(x)
      else:
        x = torch.cat((skip_connections[idx-1], x), dim=1)
        x = block(x)

    return x

In [ ]:
!pip install segmentation-models-pytorch

In [ ]:
# The training and validation loop
from segmentation_models_pytorch.losses import DiceLoss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_and_validate_UNet(model, train_loader, validation_loader, epochs, lr):
  model.to(device)

  # Using the Adam Optimizer
  optimizer = optim.Adam(model.parameters(), lr=lr)

  # Using the Dice Loss
  Loss = DiceLoss(mode='binary', from_logits=True)
  # Using Mixed Precision for faster computation
  scaler = torch.amp.GradScaler('cuda')

  training_losses = []
  validation_losses = []

  for epoch in range(epochs):
    # Training:
    model.train()
    total_training_loss = 0

    for image, label in train_loader:
      image = image.to(device, non_blocking=True)
      label = label.to(device, non_blocking=True).unsqueeze(1).float()

      optimizer.zero_grad(set_to_none=True)

      with torch.amp.autocast('cuda'):
        # Forward pass
        predicted = model.forward(image)
        # Calculating Loss
        loss = Loss(predicted, label)

      # Updating the total loss
      total_training_loss += loss.item()

      # Backpropagation with scaled gradients
      scaler.scale(loss).backward()
      # Optimizer step
      scaler.step(optimizer)
      scaler.update()

    # Printing the average training loss for that epoch:
    average_training_loss = total_training_loss/len(train_loader)
    training_losses.append(average_training_loss)
    print(f"Epoch {epoch + 1}:")
    print(f"Average Training Loss = {average_training_loss:.4f}")

    # Validation:
    model.eval()
    total_validation_loss = 0

    with torch.no_grad():
      for image, label in validation_loader:
        image = image.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True).unsqueeze(1).float()

      with torch.amp.autocast('cuda'):
        # Forward pass
        predicted = model.forward(image)
        # Calculating Loss
        loss = Loss(predicted, label)

      # Updating the total loss
      total_validation_loss += loss.item()

    # Printing the average validation loss for that epoch
    average_validation_loss = total_validation_loss/len(validation_loader)
    validation_losses.append(average_validation_loss)
    print(f"Average Validation Loss = {average_validation_loss:.4f}")
    print()

  # Plotting the training and validation loss plots
  plt.figure(figsize=(6, 8))

  plt.subplot(2, 1, 1)
  plt.plot([(i+1) for i in range(epochs)], training_losses, color='red')
  plt.ylabel("Loss")
  plt.ylim(bottom=0)
  plt.title("Average Training Loss Plot")
  plt.grid(True, alpha=0.3)

  plt.subplot(2, 1, 2)
  plt.plot([(i+1) for i in range(epochs)], validation_losses, color='blue')
  plt.ylabel("Loss")
  plt.xlabel("Epoch")
  plt.ylim(bottom=0)
  plt.title("Average Validation Loss Plot")
  plt.grid(True, alpha=0.3)

  plt.suptitle(f"Training and Validation Loss Plots\nEpochs = {epochs}, Learning Rate = {lr}")
  plt.show()

In [ ]:
# The testing loop
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def test_UNet(model, test_loader):
  model.to(device)

  # Testing:
  model.eval()

  # LLM used here for help with the smp library syntax
  tp_all, fp_all, fn_all, tn_all = [], [], [], []
  with torch.no_grad():
    for image, label in test_loader:
      image = image.to(device, non_blocking=True)
      label = label.to(device, non_blocking=True).unsqueeze(1).long()

      with torch.amp.autocast('cuda'):
        # Forward pass
        predicted = model.forward(image)

      # Calculating road probabilities from logits
      probs = torch.sigmoid(predicted)
      # Categorizing pixels by a threshold of 0.5 probability
      preds_binary = (probs >= 0.5).long()

      # Get batch-level stats
      tp, fp, fn, tn = smp.metrics.get_stats(
          preds_binary,
          label,
          mode="binary"
      )

      # Update the batch-level stats
      tp_all.append(tp.cpu())
      fp_all.append(fp.cpu())
      fn_all.append(fn.cpu())
      tn_all.append(tn.cpu())

  tp_total = torch.cat(tp_all, dim=0)
  fp_total = torch.cat(fp_all, dim=0)
  fn_total = torch.cat(fn_all, dim=0)
  tn_total = torch.cat(tn_all, dim=0)

  # Calculate the single true, dataset-wide average IoU and Dice Coefficient
  iou = smp.metrics.iou_score(tp_total, fp_total, fn_total, tn_total, reduction="micro")
  dice = smp.metrics.f1_score(tp_total, fp_total, fn_total, tn_total, reduction="micro")

  # Printing the IoU and the Dice Coefficient
  print(f"IoU = {iou:.4f}")
  print(f"Dice Coefficient = {dice:.4f}")

In [ ]:
# Initializing the UNet Model
EPOCHS = 100
LEARNING_RATE = 0.0001
CHANNELS = 3
HEIGHT = 256
WIDTH = 256
model_unet = UNet(CHANNELS, HEIGHT, WIDTH)

# Training it
# train_and_validate_UNet(model_unet, train_loader, val_loader, EPOCHS, LEARNING_RATE)

In [ ]:
# Loading the trained checkpoints of the model
model_unet.load_state_dict(torch.load('/content/Learning_Phase/Task-2/Subtask-1/Trained_Checkpoints/learningphase_task2_subtask1_unet.pth', allow_pickle=True))

In [ ]:
# Testing the UNet
test_UNet(model_unet, test_loader)

In [ ]:
# Saving the trained checkpoints
# torch.save(model_unet.state_dict(), 'learningphase_task2_subtask1_unet.pth')

In [ ]:
# Qualitative Output Demo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Setup paths
image_folder_path = '/content/Learning_Phase/Task-2/Subtask-1/Massachusetts_Roads_Processed/'
test_files  = [os.path.join(image_folder_path, 'test_batches', f)  for f in sorted(os.listdir(os.path.join(image_folder_path, 'test_batches')))]

# Select which image to view using batch index and
# file index within that batch
check_batch = 0
check_index = 24
file_path = test_files[check_batch]

with np.load(file_path, allow_pickle=True) as data:
  X_sample = data['input'][check_index]
  y_sample = data['labels'][check_index]

  # Convert and fix dimensions here (H, W, C) -> (C, H, W)
  X_tensor = torch.from_numpy(X_sample).float().permute(2, 0, 1) / 255.0

model_unet.eval()
with torch.no_grad():
  # Get the road probabilities from the model
  logits = model_unet(X_tensor.unsqueeze(0).to(device))
  probs = torch.sigmoid(logits)
  # Multiply by 255 to get confidence scores that can directly be visualized as
  # a grayscale image
  confidence_scores = (probs.squeeze()*255).cpu().detach().numpy()

# Visualizing the original input image
plt.figure(figsize=(5, 5))
plt.imshow(X_sample)
plt.title('Input Image')
plt.axis('off')
plt.show()

# Visualizing the expected vs predicted segmentation masks
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(y_sample, cmap="gray")
axes[0].set_title('Expected Label Image')
axes[0].axis('off')

axes[1].imshow(confidence_scores, cmap="gray", vmin=0, vmax=255)
axes[1].set_title('Predicted Label Image')
axes[1].axis('off')

plt.suptitle("Expected vs Predicted Segmentation:\n(Ground Truth downscaled from 1500 x 1500 to 256 x 256)\n")

plt.tight_layout()
plt.show()

## **Attention UNet Architecture:**

In [ ]:
# Attention UNet Architecture

class Attention_UNet(nn.Module):
  def __init__(self, channels, height, width):
    super().__init__()

    # Initializing the UNet parameters
    self.channels = channels
    self.height = height
    self.width = width

    # Defining the blocks of the Contracting Path:
    self.contract = nn.ModuleList([
        # Block 1
        nn.Sequential(
            nn.Conv2d(self.channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        ),
        # Block 2
        nn.Sequential(
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        ),
        # Block 3
        nn.Sequential(
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        ),
        # Block 4
        nn.Sequential(
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU()
        )
    ])

    # Defining the blocks of the Expanding Path:
    self.expand = nn.ModuleList([
        # Block 4
        nn.Sequential(
            nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2),
            nn.BatchNorm2d(256),
            nn.ReLU()
        ),
        # Block 3
        nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2),
            nn.BatchNorm2d(128),
            nn.ReLU()
        ),
        # Block 2
        nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),
            nn.BatchNorm2d(64),
            nn.ReLU()
        ),
        # Block 1
        nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 1, kernel_size=1)
        )
    ])

    # Defining the projections for the gating signals:
    self.g_projections = nn.ModuleList([
        nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=1, stride=1),
            nn.BatchNorm2d(128)
        ),
        nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=1, stride=1),
            nn.BatchNorm2d(64)
        ),
        nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=1, stride=1),
            nn.BatchNorm2d(32)
        )
    ])

    # Defining the projections for the skip connections:
    self.x_pojections = nn.ModuleList([
        nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=1, stride=1),
            nn.BatchNorm2d(128)
        ),
        nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=1, stride=1),
            nn.BatchNorm2d(64)
        ),
        nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=1, stride=1),
            nn.BatchNorm2d(32)
        )
    ])

    # Defining the attention blocks:
    self.attention = nn.ModuleList([
        nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(128, 1, kernel_size=1, stride=1),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        ),
        nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(64, 1, kernel_size=1, stride=1),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        ),
        nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(32, 1, kernel_size=1, stride=1),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
    ])

  def forward(self, x):
    skip_connections = []

    # Passing the input through the contracting path
    # and storing the skip connection information
    for block in self.contract:
      x = block(x)
      skip_connections.append(x)

    # Reversing the skip connections
    skip_connections = list(reversed(skip_connections))

    # Passing the encoded input through the expanding path,
    # calculating additive attention, and concatenating
    for idx, block in enumerate(self.expand):
      if idx == 3:
        x = block(x)
      else:
        g = block(x)
        g_proj = self.g_projections[idx](g)
        x_proj = self.x_pojections[idx](skip_connections[idx+1])
        x = x_proj + g_proj
        x = self.attention[idx](x) * skip_connections[idx+1]
        x = torch.cat((g, x), dim=1)

    return x

In [ ]:
# Initializing the UNet Model
EPOCHS = 100
LEARNING_RATE = 0.0001
CHANNELS = 3
HEIGHT = 256
WIDTH = 256
model_attention_unet = Attention_UNet(CHANNELS, HEIGHT, WIDTH)

# Training it
# train_and_validate_UNet(model_attention_unet, train_loader, val_loader, EPOCHS, LEARNING_RATE)

In [ ]:
# Loading the trained checkpoints of the model
model_attention_unet.load_state_dict(torch.load('/content/Learning_Phase/Task-2/Subtask-1/Trained_Checkpoints/learningphase_task2_subtask1_attention_unet.pth', allow_pickle=True))

In [ ]:
# Testing the UNet
test_UNet(model_attention_unet, test_loader)

In [ ]:
# Saving the trained checkpoints
# torch.save(model_attention_unet.state_dict(), 'learningphase_task2_subtask1_attention_unet.pth')

In [ ]:
# Qualitative Output Demo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Setup paths
image_folder_path = '/content/Learning_Phase/Task-2/Subtask-1/Massachusetts_Roads_Processed/'
test_files  = [os.path.join(image_folder_path, 'test_batches', f)  for f in sorted(os.listdir(os.path.join(image_folder_path, 'test_batches')))]

# Select which image to view using batch index and
# file index within that batch
check_batch = 0
check_index = 24
file_path = test_files[check_batch]

with np.load(file_path, allow_pickle=True) as data:
  X_sample = data['input'][check_index]
  y_sample = data['labels'][check_index]

  # Convert and fix dimensions here (H, W, C) -> (C, H, W)
  X_tensor = torch.from_numpy(X_sample).float().permute(2, 0, 1) / 255.0

model_attention_unet.eval()
with torch.no_grad():
  # Get the road probabilities from the model
  logits = model_attention_unet(X_tensor.unsqueeze(0).to(device))
  probs = torch.sigmoid(logits)
  # Multiply by 255 to get confidence scores that can directly be visualized as
  # a grayscale image
  confidence_scores = (probs.squeeze()*255).cpu().detach().numpy()

# Visualizing the original input image
plt.figure(figsize=(5, 5))
plt.imshow(X_sample)
plt.title('Input Image')
plt.axis('off')
plt.show()

# Visualizing the expected vs predicted segmentation masks
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(y_sample, cmap="gray")
axes[0].set_title('Expected Label Image')
axes[0].axis('off')

axes[1].imshow(confidence_scores, cmap="gray", vmin=0, vmax=255)
axes[1].set_title('Predicted Label Image')
axes[1].axis('off')

plt.suptitle("Expected vs Predicted Segmentation:\n(Ground Truth downscaled from 1500 x 1500 to 256 x 256)\n")

plt.tight_layout()
plt.show()